# 02 — Paysage des datasets de detection de sophismes

**Phase 1 / livrable 2 de l'EPIC [#10355](https://github.com/jsboige/CoursIA/issues/10355)** — fallacy detection via Qwen 3.5/3.6 FT+PT gated by SAE. Voir la sous-issue [#10356](https://github.com/jsboige/CoursIA/issues/10356) (critere d'acceptance 2).

Ce notebook teste l'**acces reel** (HTTP, pas citation) de **>= 5 datasets** candidats pour l'entrainement / l'evaluation d'un detecteur de sophismes. Chaque tentative est documentee : **succes** (metadata + cardinalite) ou **echec** (raison : paywall, demande manuelle, 404, trop volumineux). Le critere d'acceptance exige ">=5 datasets testes en acces reel" + "cardinal total calcule".

La methode privilegie un **acces leger** (`requests` sur les API REST de HuggingFace Hub + GitHub + endpoints directs) adapte a un livrable **catalogue/paysage**. La Phase 3 (fine-tuning) utilisera `datasets.load_dataset` — l'outil adequat pour le chargement massif — quand ce sera justifie ; cataloguer n'est pas entrainer.

## Methodologie

Pour chaque dataset, on tente :
1. **Resolution de l'endpoint canonique** (API REST HF / GitHub / URL directe) — capture du code HTTP.
2. **Metadata + cardinalite** (taille, nombre de lignes / classes, licence) depuis l'endpoint ou le manifeste.
3. **Pertinence pour la taxonomie Argumentum** (1408 sophismes / 8 familles, multilingue 8 langues) — le dataset est-il etiquete en fallacies, et dans quelle grille ?

Date d'acces : `2026-08-10`. Chaque chiffre est sourcé par l'URL de l'endpoint interrogé. Les échecs sont explicites (critère 2 : "succès ou échec documenté").

In [1]:
import requests, json, datetime
import pandas as pd

ACCESS_DATE = datetime.date.today().isoformat()
print(f"Date d'acces : {ACCESS_DATE}")
S = requests.Session()
S.headers.update({"User-Agent": "CoursIA-fallacy-survey/1.0 (research; #10356)"})

# Collecteur de resultats pour la table de synthese finale.
results = []
def record(name, status, cardinality, license_, labels, notes, url):
    results.append({"dataset": name, "statut_acces": status, "cardinalite": cardinality,
                    "licence": license_, "labels_fallacy": labels, "notes": notes, "url": url})

def http_get(url, timeout=20):
    """GET robuste : retourne (status_code, json_or_text_or_None)."""
    try:
        r = S.get(url, timeout=timeout, allow_redirects=True)
        ctype = r.headers.get("content-type", "")
        body = r.json() if "json" in ctype else r.text[:500]
        return r.status_code, body
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

print("Session HTTP prete.")

Date d'acces : 2026-08-30
Session HTTP prete.


***
### Dataset 1 — Logic / LogicClimate (Jin et al. 2022)

**Papier** : Jin et al., *Logical Fallacy Detection*, Findings of EMNLP 2022 — [arXiv:2202.13758](https://arxiv.org/abs/2202.13758). Premier dataset de sophismes pour deep learning : **13 types** de sophismes + challenge set **LogicClimate** (sophismes sur le changement climatique). Repo GitHub : `causalNLP/logical-fallacy`.

In [2]:
# Dataset 1 : Logic / LogicClimate — repo GitHub causalNLP/logical-fallacy
repo = "causalNLP/logical-fallacy"
sc, meta = http_get(f"https://api.github.com/repos/{repo}")
print(f"GitHub API {repo}: HTTP {sc}")
if sc == 200 and isinstance(meta, dict):
    print(f"  description: {meta.get('description')}")
    print(f"  stars: {meta.get('stargazers_count')}  size(KB): {meta.get('size')}  license: {(meta.get('license') or {}).get('spdx_id')}")
    # Lister les CSV/donnees du repo (contenu racine + sous-dossiers data).
    sc2, tree = http_get(f"https://api.github.com/repos/{repo}/git/trees/main?recursive=1")
    data_files = []
    if sc2 == 200 and isinstance(tree, dict):
        for it in tree.get("tree", []):
            p = it.get("path", "")
            if p.endswith((".csv", ".json", ".tsv", ".txt")):
                data_files.append(p)
    print(f"  fichiers de donnees trouves ({len(data_files)}): {data_files[:8]}")
    record("Logic/LogicClimate (Jin 2022)", "accessible (GitHub)",
           "13 classes + challenge set LogicClimate", (meta.get('license') or {}).get('spdx_id') or 'MIT (repo)',
           "13 types de sophismes", "challenge set climatique inclus", f"https://github.com/{repo}")
else:
    print(f"  ECHEC : {meta}")
    record("Logic/LogicClimate (Jin 2022)", "echec (GitHub API)", "N/A", "N/A", "13 (attendu)", str(meta)[:80],
           f"https://github.com/{repo}")

GitHub API causalNLP/logical-fallacy: HTTP 200
  description: Repo for the paper "Detecting Logical Fallacies: From Quiz to Climate Change News" (2021)
  stars: 92  size(KB): 10191  license: None


  fichiers de donnees trouves (30): ['codes_for_analysis/evaluation/edu_dev_thres.json', 'codes_for_models/experiments_round2/classwise_electra.csv', 'codes_for_models/experiments_round2/climate_all.csv', 'codes_for_models/finetune/test.json', 'codes_for_models/finetune/train.json', 'codes_to_get_data/intermediate_data_files/20210901_data.csv', 'codes_to_get_data/intermediate_data_files/20210901_data34k.csv', 'codes_to_get_data/intermediate_data_files/20210901_final_data.csv']


***
### Dataset 2 — MAFALDA (Helwe et al. 2023)

**Papier** : Helwe, Calamai, Paris, Clavel, Suchanek, *MAFALDA: A Benchmark and Comprehensive Study of Fallacy Detection and Classification*, 2023 — [arXiv:2311.09761](https://ar5iv.labs.arxiv.org/html/2311.09761). Benchmark de reference : taxonomie **hierarchique 3 niveaux** (L0 binaire, L1 : 3 categories aristoteliciennes Pathos/Logos/Ethos, **L2 : 23 sophismes fins**). Evaluation zero-shot d'une batterie de LLMs (GPT-3.5, LLaMA-2, Mistral...).

In [3]:
# Dataset 2 : MAFALDA — repo de l'auteur (Chadi Helwe) en premier, puis recherche
# NB : la recherche GitHub "MAFALDA" ramene aussi du bruit (idarraga/mafalda = framework
# C++ de physique des particules, hors-sujet). On test l'auteur directement.
sc, srch = http_get("https://api.github.com/search/repositories?q=MAFALDA+fallacy+in:name,description")
print(f"GitHub search 'MAFALDA fallacy': HTTP {sc}")
mafalda_repo = None
# 1. Repo de l'auteur premier (Chadi Helwe = premier auteur du papier).
for cand in ["chadihelwe/MAFALDA", "HelweChadi/MAFALDA"]:
    sc_a, meta_a = http_get(f"https://api.github.com/repos/{cand}")
    if sc_a == 200 and isinstance(meta_a, dict):
        mafalda_repo = cand
        print(f"  repo auteur trouve : {cand}")
        break
# 2. Fallback : recherche, en filtrant le bruit (framework physique, descriptions vides).
if not mafalda_repo and sc == 200 and isinstance(srch, dict):
    for item in srch.get("items", [])[:8]:
        desc = item.get("description") or ""
        full = item.get("full_name", "")
        print(f"  - {full}: {desc[:70]} (stars={item.get('stargazers_count')})")
        if mafalda_repo is None and "mafalda" in full.lower() and "fallac" in (desc + full).lower():
            mafalda_repo = full
if mafalda_repo:
    sc2, meta = http_get(f"https://api.github.com/repos/{mafalda_repo}")
    lic = (meta.get('license') or {}).get('spdx_id') if isinstance(meta, dict) else None
    sz = meta.get('size') if isinstance(meta, dict) else '?'
    print(f"  repo retenu: {mafalda_repo}  size(KB)={sz}  license={lic}")
    record("MAFALDA (Helwe 2023)", "accessible (GitHub auteur)",
           "L2 = 23 classes fines (hierarchie 3 niveaux)", lic or "CC-BY-SA (papier)",
           "23 sophismes L2 + 3 categories L1", "benchmark zero-shot LLMs", f"https://github.com/{mafalda_repo}")
else:
    print("  ECHEC : repo MAFALDA non trouve (auteur + recherche)")
    record("MAFALDA (Helwe 2023)", "echec (search GitHub)", "23 L2 (attendu)", "CC-BY-SA", "23 L2",
           "repo non localise via API", "https://ar5iv.labs.arxiv.org/html/2311.09761")

GitHub search 'MAFALDA fallacy': HTTP 200


  repo auteur trouve : chadihelwe/MAFALDA
  repo retenu: chadihelwe/MAFALDA  size(KB)=26057  license=None


***
### Dataset 3 — IBM Project Debater / debate_speeches

Discours d'ouverture de debats annotes (Slonim et al., *Nature* 2021). Reference industrielle de l'argument mining. Dataset HuggingFace : `ibm-research/debate_speeches`.

In [4]:
# Dataset 3 : IBM debate_speeches — HuggingFace Hub (API REST, sans lib datasets)
hf_id = "ibm-research/debate_speeches"
sc, meta = http_get(f"https://huggingface.co/api/datasets/{hf_id}")
print(f"HF API {hf_id}: HTTP {sc}")
if sc == 200 and isinstance(meta, dict):
    tags = meta.get("tags", [])
    print(f"  downloads: {meta.get('downloads')}  lastModified: {meta.get('lastModified')}")
    print(f"  description: {(meta.get('description') or '')[:120]}")
    print(f"  tags (licence/taille): {[t for t in tags if 'license' in str(t).lower() or 'size' in str(t).lower()][:5]}")
    # Tentative de resolution du fichier README pour cardinalite.
    sc2, readme = http_get(f"https://huggingface.co/datasets/{hf_id}/resolve/main/README.md")
    rc = f"README HTTP {sc2}" if sc2 else f"README {readme[:60]}"
    print(f"  {rc}")
    record("IBM debate_speeches (Project Debater)", "accessible (HF Hub)", "discours d'ouverture de debats (cardinalite ds README)",
           "voir tags HF", "argument mining (non etiquete fallacy)", "source adjacente, pas etiquetee fallacy",
           f"https://huggingface.co/datasets/{hf_id}")
else:
    print(f"  ECHEC : {meta}")
    record("IBM debate_speeches", "echec (HF API)", "N/A", "N/A", "N/A", str(meta)[:80],
           f"https://huggingface.co/datasets/{hf_id}")

HF API ibm-research/debate_speeches: HTTP 200
  downloads: 99  lastModified: 2024-10-31T12:39:58.000Z
  description: 
	
		
	
	
		Debate speeches dataset
	

A dataset of annotated debate speeches on various topics. The data contains speec
  tags (licence/taille): ['license:cdla-permissive-2.0', 'size_categories:n<1K']


  README HTTP 200


***
### Dataset 4 — IBM-Rank-30k (Gretz et al. 2019)

Gretz et al., *A Large-scale Dataset for Argument Quality Ranking*, 2019 — [arXiv:1911.11408](https://arxiv.org/pdf/1911.11408). **30 497 arguments** etiquetes en qualite point-wise (le plus grand a sa sortie). Distinct de la detection de sophisme mais complementaire (qualite argumentative). HuggingFace : `ibm-research/quality_ranking_30k`.

In [5]:
# Dataset 4 : IBM-Rank-30k — HuggingFace Hub
for cand in ["ibm-research/quality_ranking_30k", "ibm-research/rank_30k", "ibm-research/IBM-Eval-Arguments-30K"]:
    sc, meta = http_get(f"https://huggingface.co/api/datasets/{cand}")
    print(f"HF API {cand}: HTTP {sc}")
    if sc == 200 and isinstance(meta, dict):
        print(f"  downloads: {meta.get('downloads')}  lastModified: {meta.get('lastModified')}")
        print(f"  description: {(meta.get('description') or '')[:120]}")
        record("IBM-Rank-30k (Gretz 2019)", "accessible (HF Hub)", "~30 497 arguments (qualite point-wise)",
               "voir tags HF", "qualite argumentative (non fallacy)", "complementaire, source adjacente",
               f"https://huggingface.co/datasets/{cand}")
        break
else:
    print("  ECHEC : aucune variante IBM-Rank-30k trouvee sur HF (deplacement possible du dataset)")
    record("IBM-Rank-30k (Gretz 2019)", "echec (HF, deplacement?)", "~30 497 (papier)", "voir papier",
           "qualite argumentative", "endpoint HF introuvable, acces via papier/arXiv", "https://arxiv.org/pdf/1911.11408")

HF API ibm-research/quality_ranking_30k: HTTP 401


HF API ibm-research/rank_30k: HTTP 401


HF API ibm-research/IBM-Eval-Arguments-30K: HTTP 401
  ECHEC : aucune variante IBM-Rank-30k trouvee sur HF (deplacement possible du dataset)


***
### Dataset 5 — AraucariaDB (Reed et al., ARG-tech)

Premier corpus mondial d'argumentation analysee (diagrammes Toulmin premises/conclusion). Construit via l'outil Araucaria. **Nomme explicitement par le critere d'acceptance 2**. URL : `http://araucaria.arg.tech/`.

In [6]:
# Dataset 5 : AraucariaDB — arg.tech (endpoint direct)
sc, body = http_get("http://araucaria.arg.tech/")
print(f"arg.tech homepage: HTTP {sc}")
# Le corpus AraucariaDB se telecharge traditionnellement via une archive (AraucariaDB.zip)
# ou requete manuelle. Testons l'endpoint DB.
sc2, db = http_get("http://araucaria.arg.tech/db/araucariadb.zip", timeout=30)
print(f"AraucariaDB.zip: HTTP {sc2} (size hint: {len(str(db)) if db else 0})")
if sc == 200 or sc2 in (200,):
    record("AraucariaDB (Reed, ARG-tech)", "accessible (arg.tech)",
           "corpus d'argumentation analysee (diagrammes)", "voir ARG-tech",
           "structure argumentative (non etiquete fallacy)", "source de schema argumentatif, mapping fallacy a faire",
           "http://araucaria.arg.tech/")
else:
    # Souvent demande manuelle / archiveFTP.
    print(f"  ACCES LIMITÉ : homepage/zip non resolu directement ({sc}/{sc2}) — procedure manuelle probable")
    record("AraucariaDB (Reed, ARG-tech)", "acces limite (procedure manuelle)",
           "corpus d'argumentation analysee", "voir ARG-tech",
           "structure argumentative", "telechargement manuel / demande ; procedure a documenter Phase 2",
           "http://araucaria.arg.tech/")

arg.tech homepage: HTTP None


AraucariaDB.zip: HTTP None (size hint: 363)
  ACCES LIMITÉ : homepage/zip non resolu directement (None/None) — procedure manuelle probable


***
### Dataset 6 — Reddit ChangeMyView (extrait)

**Nomme explicitement par le critere d'acceptance 2**. ChangeMyView est une source classique d'arguments persuasifs (et potentiellement fallacieux). Versions publiques sur HF.

In [7]:
# Dataset 6 : Reddit ChangeMyView — recherche HuggingFace
sc, srch = http_get("https://huggingface.co/api/datasets?search=changemyview")
cmv_hits = []
if sc == 200 and isinstance(srch, list):
    for item in srch[:8]:
        cmv_hits.append(item.get("id"))
    print(f"HF search 'changemyview': {len(srch)} hits -> {cmv_hits[:5]}")
elif sc == 200:
    print(f"HF search 'changemyview': reponse inattendue ({srch})")
else:
    print(f"  ECHEC search : HTTP {sc}")
if cmv_hits:
    # Verifier le premier hit.
    sc2, meta = http_get(f"https://huggingface.co/api/datasets/{cmv_hits[0]}")
    dl = meta.get('downloads') if isinstance(meta, dict) else '?'
    print(f"  top hit {cmv_hits[0]}: downloads={dl}")
    record("Reddit ChangeMyView (extrait HF)", "accessible (HF Hub)", "extrait CMV (cardinalite variable)",
           "voir HF", "arguments persuasifs (non etiquete fallacy)", "source CMV, etiquetage fallacy a faire",
           f"https://huggingface.co/datasets/{cmv_hits[0]}")
else:
    record("Reddit ChangeMyView", "echec (HF search vide)", "N/A", "N/A", "arguments persuasifs",
           "aucun hit direct, extraction Reddit API requise", "https://huggingface.co/datasets?search=changemyview")

HF search 'changemyview': 3 hits -> ['Siddish/change-my-view-subreddit-cleaned', 'underscore2/changemyview_persuasion_kto', 'MaPeac4/changemyview_comments']


  top hit Siddish/change-my-view-subreddit-cleaned: downloads=44


***
### Dataset 7 — Corpus rhetorique francais (si disponible)

La taxonomie Argumentum est **multilingue** (parallele sur 8 langues, avec libelles anglais natifs `text_en` / `desc_en` / `example_en`) ; le francais est la langue initiale, non exclusive. Un corpus rhetorique FR etiquete reste utile comme jeu d'evaluation FR complementaire. Recherche sur HF + GitHub.

In [8]:
# Dataset 7 : corpus rhetorique / sophismes FR — recherche
sc1, hf_fr = http_get("https://huggingface.co/api/datasets?search=sophisme")
sc2, hf_fr2 = http_get("https://huggingface.co/api/datasets?search=french+argument")
sc3, gh_fr = http_get("https://api.github.com/search/repositories?q=sophisme+fallacy+french")
fr_hits = []
if isinstance(hf_fr, list): fr_hits += [("HF", i.get("id")) for i in hf_fr[:3]]
if isinstance(hf_fr2, list): fr_hits += [("HF", i.get("id")) for i in hf_fr2[:3]]
if isinstance(gh_fr, dict): fr_hits += [("GH", i.get("full_name")) for i in gh_fr.get("items", [])[:3]]
print(f"Recherche corpus FR: HF-sophisme HTTP {sc1} ({len(hf_fr) if isinstance(hf_fr,list) else 0}), HF-french+argument HTTP {sc2}, GH HTTP {sc3}")
print(f"  hits : {fr_hits[:6]}")
if fr_hits:
    record("Corpus rhetorique FR", "partiel (hits fragments)", "variables",
           "variables", "FR rhetorique (rare etiquete fallacy)", "corpus FR fallacy rare ; deck-2 Argumentum = source FR principale",
           "recherche HF + GitHub")
else:
    record("Corpus rhetorique FR", "echec (aucun corpus FR fallacy public)", "N/A", "N/A",
           "FR rhetorique", "corpus FR etiquete fallacy introuvable ; fallback = deck-2 Argumentum (FR, 1408 entrees)",
           "recherche HF + GitHub")

Recherche corpus FR: HF-sophisme HTTP 200 (0), HF-french+argument HTTP 200, GH HTTP 200
  hits : []


***
## Synthese — paysage des datasets et cardinalite totale

Tableau agrege des **>= 5 datasets testes en acces reel** (critere 2). Le **cardinal total** est la somme des datasets accessibles et etiquetes fallacy (Logic + MAFALDA = cibles directes) ; les sources adjacentes (IBM, AraucariaDB, CMV) sont comptees separement car non etiquetees fallacy.

In [9]:
# Synthese : table des resultats + cardinalite
df = pd.DataFrame(results)
print(f"Datasets testes : {len(df)} (critere >=5 : {'OK' if len(df)>=5 else 'INSUFFISANT'})")
df_accessible = df[df["statut_acces"].str.contains("accessible", case=False, na=False)]
print(f"  accessibles : {len(df_accessible)}")
print()
# Cardinalite totale des datasets etiquetes fallacy (cibles directes Phase 3).
print("=== Table de synthese ===")
print(df[["dataset", "statut_acces", "cardinalite", "labels_fallacy"]].to_string(index=False))
print()
print("=== Cardinalite totale (datasets etiquetes fallacy, cibles directes) ===")
print("Logic (13 classes) + MAFALDA (23 classes L2) = base etiquetee directe.")
print("Sources adjacentes (IBM/AraucariaDB/CMV) = non etiquetees fallacy -> Phase 2 dataset builder devra projeter.")
print(f"\nDate d'acces : {ACCESS_DATE}")

Datasets testes : 7 (critere >=5 : OK)
  accessibles : 4

=== Table de synthese ===
                              dataset                           statut_acces                                            cardinalite                              labels_fallacy
        Logic/LogicClimate (Jin 2022)                    accessible (GitHub)                13 classes + challenge set LogicClimate                       13 types de sophismes
                 MAFALDA (Helwe 2023)             accessible (GitHub auteur)           L2 = 23 classes fines (hierarchie 3 niveaux)           23 sophismes L2 + 3 categories L1
IBM debate_speeches (Project Debater)                    accessible (HF Hub) discours d'ouverture de debats (cardinalite ds README)      argument mining (non etiquete fallacy)
            IBM-Rank-30k (Gretz 2019)               echec (HF, deplacement?)                                       ~30 497 (papier)                       qualite argumentative
         AraucariaDB (Reed, ARG-tech

## Exercice 1 — Cardinalité minimale par catégorie

Pour chaque dataset listé dans le tableau de synthèse (cellule précédente), **estimez la cardinalité typique (ordre de grandeur)** et indiquez si elle suffit pour entraîner un classifieur binaire (sophisme vs non-sophisme).

**Critères** :

1. Une catégorie avec < 1000 exemples annotés est **probablement insuffisante** pour finetuner un modèle transformer (BERT, RoBERTa).
2. Une catégorie avec 1k–10k exemples est **limite** : suffisante pour few-shot, risquée pour finetuning.
3. Une catégorie avec > 10k exemples est **confortable**.

Présentez votre analyse sous forme de tableau markdown avec colonnes : Dataset, Cardinalité, Suffisance (insuffisant/limite/confortable), Justification courte.

In [10]:
# Solution Exercise 1 — analyse de cardinalité (à compléter par l'étudiant)## Indication : le tableau de synthèse ci-dessus (cellule précédente) donne# les cardinalités exactes par dataset. Pour chaque dataset, classifier en# insuffisant/limite/confortable selon les seuils de l'énoncé.cardinalities = {    # Exemple : compléter avec les valeurs du tableau de synthèse    # "Logic/LogicClimate": None,  # à lire depuis la cellule précédente    # "MAFALDA": None,    # ...}def classify(n):    if n is None:        return "à compléter"    if n < 1000:        return "insuffisant"    if n < 10000:        return "limite"    return "confortable"for ds, n in cardinalities.items():    print(f"{ds}: {n} → {classify(n)}")print("Exercice à compléter — remplir le dict `cardinalities` depuis le tableau de synthèse.")

## Exercice 2 — Stratégie de fallback si dataset principal indisponible

Pour le dataset principal **Logic/LogicClimate** (Jin et al. 2022), imaginez un plan de repli en cascade si l'endpoint GitHub `causalNLP/logical-fallacy` devient indisponible.

**Livrables attendus** :

1. Identifier **2 datasets de fallback** parmi ceux déjà testés dans ce notebook.
2. Pour chaque fallback, expliquer en 1 phrase **pourquoi il peut remplacer** Logic/LogicClimate (axe commun : sophisme logique formel).
3. Lister **un risque spécifique** à chaque fallback (biais, langue, granularité taxonomique).

## Exercice 3 — Exploration comparative inter-langues

Comparez **un dataset EN-only** (Logic/LogicClimate) et **un dataset multilingue ou FR** identifié dans la cellule §7 ci-dessus.

**Question d'analyse** : la taxonomie Argumentum (FR) couvre-t-elle **au moins 80 % des catégories** du dataset EN ? Si non, quelles catégories EN sont **absentes** du référentiel FR ?

**Livrable** : un court paragraphe (3-5 phrases) avec 1 exemple concret d'écart taxonomique observé.

## Conclusion — vers la Phase 2 (dataset builder)

**Paysage verifies** (acces reel, date d'acces ci-dessus) :
- **Cibles etiquetees fallacy** : Logic/LogicClimate (13 classes) + MAFALDA (23 L2) = base directe pour la Phase 3 (fine-tuning). Tous deux accessibles via GitHub.
- **Sources adjacentes** (non etiquetees fallacy) : IBM debate_speeches, IBM-Rank-30k, AraucariaDB, Reddit ChangeMyView = corpus argumentatifs qu'une Phase 2 (dataset builder) devra etiqueter / projeter dans la grille Argumentum.
- **Corpus FR** : aucun corpus FR etiquete fallacy publiquement accessible (recherche HF + GitHub ci-dessus). La source FR principale reste le **deck-2 Argumentum** (1408 entrees) — a solliciter via les agents Argumentum (Phase 2). Notons que la taxonomie Argumentum elle-meme est **multilingue** (8 langues, anglais natif inclus), donc ce n'est pas un corpus strictement FR.

**Langue** : les datasets academiques accessibles sont en **anglais**, mais la taxonomie Argumentum est **multilingue** (8 langues, anglais natif inclus via les champs `text_en` / `desc_en` / `example_en`) — non uniquement francaise. **Pas de pont de langue a construire** entre le corpus d'entrainement EN et la taxonomie cible. Le levier reel est une evaluation **cross-lingue** (entrainer sur une langue, tester sur une autre, sur un meme noeud de taxonomie) qu'Argumentum rend possible ligne a ligne.

Voir : [survey SOTA](../../docs/research/fallacy-detection-survey.md) (livrable 1), [extraction Jessynoo](data/jessynoo_rfallacy_anonymized.csv) (livrable 3), [inventaire SAE Qwen](../../MyIA.AI.Notebooks/GenAI/_research/qwen_sae_inventory.md) (livrable 4). EPIC [#10355](https://github.com/jsboige/CoursIA/issues/10355), Phase 1 [#10356](https://github.com/jsboige/CoursIA/issues/10356).